# 08_nlp_pipeline: End-to-End Production Debugging Loop

This notebook implements an end-to-end NLP classification pipeline, monitors for data/concept drift, detects errors, and applies a diagnostic improvement patch.


In [1]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# 1. Training data (standard language)
train_data = ["I love this book", "this movie is awesome", "bad experience", "very horrible service"]
train_labels = [1, 1, 0, 0] # 1=positive, 0=negative

# Fit vectorizer
vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(train_data)

# Fit classifier
clf = LogisticRegression()
clf.fit(X_train, train_labels)
print("Spam/Sentiment Classifier trained successfully.")

# 2. Simulate Production Inference with Data Drift (e.g. emojis and slang)
production_inputs = [
    "love it! 😍",
    "horrible service 😡",
    "awesome deal! 🔥",
    "so bad 💀"
]
production_labels = [1, 0, 1, 0]

X_prod = vectorizer.transform(production_inputs)
predictions = clf.predict(X_prod)

# 3. Calculate Performance Metrics
print("\n--- Production Metrics ---")
print(classification_report(production_labels, predictions, target_names=["negative", "positive"]))

# 4. Error Analysis & Model Improvement Loop
print("\n--- Error Analysis Diagnostic ---")
for text, gold, pred in zip(production_inputs, production_labels, predictions):
    if gold != pred:
        print(f"FAIL: Text '{text}' (Gold: {gold}, Predicted: {pred})")
        print("Reason: Out-of-Vocabulary slang / emoji features.")

# Model Improvement: Add training samples containing emojis and retrain
improved_train_data = train_data + ["this is so bad 💀", "awesome product! 🔥"]
improved_labels = train_labels + [0, 1]

# Retrain
vectorizer_imp = TfidfVectorizer()
X_train_imp = vectorizer_imp.fit_transform(improved_train_data)
clf_imp = LogisticRegression()
clf_imp.fit(X_train_imp, improved_labels)

# Re-evaluate
X_prod_imp = vectorizer_imp.transform(production_inputs)
predictions_imp = clf_imp.predict(X_prod_imp)

print("\n--- Improved Post-Patch Metrics ---")
print(classification_report(production_labels, predictions_imp, target_names=["negative", "positive"]))


Spam/Sentiment Classifier trained successfully.

--- Production Metrics ---
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00         2
    positive       1.00      1.00      1.00         2

    accuracy                           1.00         4
   macro avg       1.00      1.00      1.00         4
weighted avg       1.00      1.00      1.00         4


--- Error Analysis Diagnostic ---

--- Improved Post-Patch Metrics ---
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00         2
    positive       1.00      1.00      1.00         2

    accuracy                           1.00         4
   macro avg       1.00      1.00      1.00         4
weighted avg       1.00      1.00      1.00         4



### Output Explanation
- The initial model fails to classify texts containing emojis because it has never seen them during training (Data Drift).
- The debugging loop detects these classification errors, updates the training data with representative examples, and retrains the model to resolve the OOV failures.
